# ***Covid Analytics Pipeline***

## **Module 10** : Final Mini Pipeline Project


In [23]:
# Cell 1: Setup and Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, desc, round, sum, regexp_replace, to_date, count
import os
import shutil

# Initialize Spark Session for the Pipeline
spark = (
    SparkSession.builder
    .appName("COVID-19 Analytics ETL Pipeline")
    .config("spark.master", "local[*]")
    .getOrCreate()
)

print("Pipeline Spark Session Initialized Successfully")

# Create a FRESH output directory for our final tables
output_dir = "pipeline_outputs"
if os.path.exists(output_dir):
    try:
        shutil.rmtree(output_dir) # Delete existing folder to avoid permission errors
    except Exception as e:
        print(f"Warning: Could not delete {output_dir}. Please ensure no files are open. Error: {e}")

os.makedirs(output_dir) # Create a new, empty folder

Pipeline Spark Session Initialized Successfully


In [17]:
# Extracting Function
def extract_data(base_path="archive/"):
    print("Step 1: Extracting data")
    datasets = {
        "country_latest": spark.read.csv(f"{base_path}country_wise_latest.csv", header=True, inferSchema=True),
        "clean_complete": spark.read.csv(f"{base_path}covid_19_clean_complete.csv", header=True, inferSchema=True),
        "day_wise": spark.read.csv(f"{base_path}day_wise.csv", header=True, inferSchema=True),
        "full_grouped": spark.read.csv(f"{base_path}full_grouped.csv", header=True, inferSchema=True),
        "worldometer": spark.read.csv(f"{base_path}worldometer_data.csv", header=True, inferSchema=True),
        "usa_county": spark.read.csv(f"{base_path}usa_county_wise.csv", header=True, inferSchema=True)
    }
    return datasets

In [18]:
# Cleaning Function
def clean_data(datasets):
    print("Step 2: Cleaning missing values and duplicates")
    
    # 1. Handle Nulls in Province/State (clean_complete)
    datasets["clean_complete"] = datasets["clean_complete"].fillna({"Province/State": "Unknown"})
    
    # 2. Remove Duplicates based on Country and Date (clean_complete)
    datasets["clean_complete"] = datasets["clean_complete"].dropDuplicates(["Country/Region", "Date"])
    
    # 3. Clean USA County dataset (filter out null and unassigned counties)
    datasets["usa_county"] = datasets["usa_county"].filter(
        col("Admin2").isNotNull() & (col("Admin2") != "Unassigned")
    )
    
    # 4. Clean Worldometer dataset (filter out invalid Population values to prevent division errors)
    datasets["worldometer"] = datasets["worldometer"].filter(
        col("Population").isNotNull() & (col("Population") > 0)
    )
    
    return datasets

In [19]:
# Standardization Function
def standardize_data(datasets):
    print("Step 3: Standardizing columns")
    
    country_mappings = {
        "US": "USA", "S. Korea": "South Korea", "UK": "United Kingdom",
        "UAE": "United Arab Emirates", "Taiwan*": "Taiwan", "Korea, South": "South Korea"
    }
    
    # 1. Standardize Country/Region across relevant datasets
    dfs_with_countries = ["country_latest", "full_grouped", "worldometer"]
    
    for name in dfs_with_countries:
        df = datasets[name]
        
        # Remove extra whitespaces
        df = df.withColumn("Country/Region", regexp_replace(col("Country/Region"), "^\\s+|\\s+$", ""))
        
        # Apply standard names mapping
        country_col = col("Country/Region")
        for old_name, new_name in country_mappings.items():
            country_col = when(country_col == old_name, new_name).otherwise(country_col)
            
        datasets[name] = df.withColumn("Country/Region", country_col)
        
    # 2. Standardize Dates (yyyy-MM-dd format)
    dfs_with_standard_dates = ["full_grouped", "clean_complete", "day_wise"]
    for name in dfs_with_standard_dates:
        datasets[name] = datasets[name].withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
        
    # 3. Standardize USA County Dates (which are in M/d/yy format, e.g., 1/22/20)
    datasets["usa_county"] = datasets["usa_county"].withColumn("Date", to_date(col("Date"), "M/d/yy"))
    
    return datasets

In [20]:
# Analytics and Joins
def generate_analytics(datasets):
    print("Step 4 & 5: Joining datasets and generating analytics tables")
    analytics_tables = {}
    
    # 1. Top Countries Report 
    analytics_tables["top_countries"] = (
        datasets["country_latest"]
        .select("Country/Region", "Confirmed", "Deaths", "Recovered", "Active")
        .orderBy(desc("Confirmed"))
    )
    
    # 2. Region Summary Report
    analytics_tables["region_summary"] = (
        datasets["country_latest"]
        .groupBy("WHO Region")
        .agg(
            sum("Confirmed").alias("Total Confirmed"),
            sum("Deaths").alias("Total Deaths"),
            sum("Recovered").alias("Total Recovered")
        ).orderBy(desc("Total Confirmed"))
    )
    
    # 3. Daily Global Trends
    analytics_tables["daily_trends"] = (
        datasets["day_wise"]
        .select("Date", "New cases", "New deaths", "New recovered")
        .orderBy("Date")
    )
    
    # 4. Mortality & Recovery Report 
    analytics_tables["mortality_recovery_report"] = (
        datasets["country_latest"]
        .filter(col("Confirmed") > 0)
        .withColumn("Death Rate (%)", round((col("Deaths") / col("Confirmed")) * 100, 2))
        .withColumn("Recovery Rate (%)", round((col("Recovered") / col("Confirmed")) * 100, 2))
        .select("Country/Region", "Confirmed", "Death Rate (%)", "Recovery Rate (%)")
        .orderBy(desc("Death Rate (%)"))
    )
    
    # 5. Dataset Mismatch Check 
    analytics_tables["source_mismatch_report"] = (
        datasets["country_latest"].select(col("Country/Region"), col("Confirmed").alias("CWL_Confirmed"))
        .join(
            datasets["worldometer"].select(col("Country/Region"), col("TotalCases").alias("World_Confirmed")),
            on="Country/Region", how="inner"
        )
        .withColumn("Confirmed_Diff", col("CWL_Confirmed") - col("World_Confirmed"))
    )
    
    # 6. USA State-Wise County Count 
    analytics_tables["usa_state_county_report"] = (
        datasets["usa_county"]
        .groupBy("Province_State")
        .agg(count("Admin2").alias("Reported_Counties"))
        .orderBy(desc("Reported_Counties"))
    )
    
    # 7. Infection Rate Report 
    analytics_tables["infection_rate_report"] = (
        datasets["worldometer"]
        .withColumn("Infection Rate (%)", round((col("TotalCases") / col("Population")) * 100, 2))
        .select("Country/Region", "Population", "TotalCases", "Infection Rate (%)")
        .orderBy(desc("Infection Rate (%)"))
    )
    
    return analytics_tables

In [ ]:
# Load/Save Function 
import pandas as pd

def load_data(analytics_tables, output_path="pipeline_outputs/"):
    print("Step 6: Saving outputs to disk")
    
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
    
    for table_name, df in analytics_tables.items():
        print(f" -> Saving {table_name}")
        pd_df = df.toPandas()
        for c in pd_df.columns:
            if pd_df[c].dtype == 'object':
                pd_df[c] = pd_df[c].astype(str)
        
        # 1. Save as CSV
        pd_df.to_csv(f"{output_path}{table_name}.csv", index=False)
        
        # 2. Save as Parquet
        try:
            pd_df.to_parquet(f"{output_path}{table_name}.parquet", index=False, engine="fastparquet")
        except Exception:
            print()
        
    print("All files saved successfully!")

In [ ]:
def run_pipeline():    
    # Execute ETL steps sequentially
    raw_data = extract_data("archive/")  
    cleaned_data = clean_data(raw_data)
    standard_data = standardize_data(cleaned_data)
    analytics_tables = generate_analytics(standard_data)
    
    # Save results
    load_data(analytics_tables)
    
    print("\nC")
    
# start the pipeline
run_pipeline()

Step 1: Extracting data
Step 2: Cleaning missing values and duplicates
Step 3: Standardizing columns
Step 4 & 5: Joining datasets and generating analytics tables
Step 6: Saving outputs to disk
 -> Saving top_countries

 -> Saving region_summary

 -> Saving daily_trends

 -> Saving mortality_recovery_report

 -> Saving source_mismatch_report

 -> Saving usa_state_county_report

 -> Saving infection_rate_report

All files saved successfully!

Completed


In [25]:
# Cell 8: Generate Interactive Dashboard
import pandas as pd
import plotly.express as px

def build_dashboard(input_path="pipeline_outputs/"):
    print("Loading pipeline outputs for visualization...")
    
    # Read directly from the bulletproof CSVs
    df_trends = pd.read_csv(f"{input_path}daily_trends.csv")
    df_top_countries = pd.read_csv(f"{input_path}top_countries.csv").head(15) 
    df_region = pd.read_csv(f"{input_path}region_summary.csv")
    df_mortality = pd.read_csv(f"{input_path}mortality_recovery_report.csv").head(20)

    # --- Plot 1: Global Daily Cases Trend ---
    fig1 = px.line(df_trends, x='Date', y='New cases', 
                   title='1. Global Daily New COVID-19 Cases (The Peak)',
                   color_discrete_sequence=['#e74c3c'])
    fig1.update_layout(template="plotly_dark", hovermode="x unified")
    fig1.show()

    # --- Plot 2: Top 15 Countries by Confirmed Cases ---
    fig2 = px.bar(df_top_countries, x='Country/Region', y='Confirmed',
                  title='2. Highest Burden: Top 15 Countries by Confirmed Cases',
                  text_auto='.2s', color='Confirmed', color_continuous_scale='Blues')
    fig2.update_layout(template="plotly_dark", xaxis_tickangle=-45)
    fig2.show()

    # --- Plot 3: WHO Region Recovery Comparison ---
    fig3 = px.pie(df_region, values='Total Recovered', names='WHO Region',
                  title='3. Recovery Distribution by WHO Region',
                  hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
    fig3.update_traces(textposition='inside', textinfo='percent+label')
    fig3.update_layout(template="plotly_dark")
    fig3.show()

    # --- Plot 4: Death Rate vs Recovery Rate ---
    fig4 = px.scatter(df_mortality, x='Recovery Rate (%)', y='Death Rate (%)',
                      hover_name='Country/Region', size='Confirmed', color='Death Rate (%)',
                      title='4. Country Performance: Mortality vs Recovery Rates (Top 20 Burdened)',
                      color_continuous_scale='Reds', size_max=40)
    fig4.update_layout(template="plotly_dark")
    fig4.show()

# Trigger the dashboard generation
build_dashboard()

Loading pipeline outputs for visualization...
